# За вкъщи: упражнение 2, част 4

**Видове данни, представяне и сходство**

1. Пишете кода в съответните клетки на тази тетрадка.
2. Преди предаване рестартирайте ядрото и изпълнете цялата тетрадка.
3. Изпратете я на mkorudzhiev@uni-sofia.bg с тема "Уп. 2, част 4 - <Вашето име>, <Фак. номер>".

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, pairwise_distances
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

DATA_DIR = Path("data")


def tokenize(text):
    """Малки букви; всеки символ, който не е буква или цифра, става интервал."""
    cleaned = "".join(symbol.lower() if symbol.isalnum() else " " for symbol in text)
    return cleaned.split()


# SMS съобщенията, разделени както в упражнението
sms = pd.read_csv(DATA_DIR / "sms_spam.csv").drop_duplicates("text").reset_index(drop=True)
train_texts, test_texts, y_train, y_test = train_test_split(
    sms["text"], sms["label"], test_size=0.2, random_state=42, stratify=sms["label"]
)

# Кварталите с координати в километри от центъра
CENTER_LAT, CENTER_LON = 42.6977, 23.3219
neighborhoods = pd.read_csv(DATA_DIR / "sofia_neighborhoods.csv")
neighborhoods["east_km"] = (neighborhoods["lon"] - CENTER_LON) * 111 * np.cos(np.radians(CENTER_LAT))
neighborhoods["north_km"] = (neighborhoods["lat"] - CENTER_LAT) * 111

# Наеманията на велосипеди с времето навън като текст, разделени по време
bikes = pd.read_csv(DATA_DIR / "bike_hour.csv")
WEATHER = {1: "ясно", 2: "облачно", 3: "слаб валеж", 4: "силен валеж"}
rides = pd.DataFrame({
    "time": pd.to_datetime(bikes["dteday"] + " " + bikes["hr"].astype(str).str.zfill(2) + ":00"),
    "hour": bikes["hr"],
    "workingday": bikes["workingday"],
    "year": bikes["yr"],
    "weather": bikes["weathersit"].map(WEATHER),
    "count": bikes["cnt"],
})
rides_train = rides[rides["time"] < "2012-07-01"]
rides_test = rides[rides["time"] >= "2012-07-01"]

# Обявите с квартала и разстоянието до центъра, както в упражнението
apartments = pd.read_csv(DATA_DIR / "sofia_apartments.csv")
apartments["neighborhood"] = (apartments["address"].str.split(", ").str[1]
                              .str.replace("ж.к. ", "").str.replace("кв. ", ""))
located = apartments.merge(neighborhoods, on="neighborhood", how="left")
located["distance_km"] = np.hypot(located["east_km"], located["north_km"])

print("SMS:", len(train_texts), "за обучение и", len(test_texts), "за проверка")
print("Квартали:", len(neighborhoods), "| часове с наемания:", len(rides), "| обяви:", len(located))

## Задача 1. Почти еднакви съобщения

В упражнението бяха махнати само съвсем еднаквите съобщения. Спамът обаче
идва на вълни: една и съща реклама с малки разлики, например с друг
телефонен номер.

Представете всяко съобщение като множеството от думите му
(`set(tokenize(text))`). **Жакардовата близост** на две
множества е J(A, B) = |A ∩ B| / |A ∪ B|: делът на общите думи сред всички
думи, които са в поне едно от двете съобщения.

1. За всяко тестово съобщение намерете най-голямата му Жакардова близост до
   тренировъчно съобщение. Колко тестови съобщения имат близост поне 0.8?
   Колко от тях са спам и колко legit?
2. Покажете двойката с най-голяма близост под 1. Защо `drop_duplicates` не
   я е махнал? Има ли двойки с близост точно 1 и с какво се различават
   текстовете им?

<details><summary>Подсказка</summary>

`A & B` и `A | B` дават сечението и обединението на две множества. Цикъл по
всички двойки отнема няколко секунди. По-бързо е с двоични матрици: ако
`B_test` и `B_train` имат 1 за всяка дума в съобщението, `B_test @ B_train.T`
дава броя общи думи за всяка двойка, а обединението е сборът от броя думи в
двете съобщения минус сечението. Тренировъчният речник има
`len(set().union(*train_sets))` думи.

</details>

In [ ]:
# train_sets = [set(tokenize(text)) for text in train_texts]
# test_sets = [set(tokenize(text)) for text in test_texts]
# best = ...  # най-голямата близост за всяко тестово съобщение
# closest = ...  # номерът на най-близкото тренировъчно съобщение
# print("Близост поне 0.8:", ..., "| спам:", ..., "| legit:", ...)
#
# vocabulary_size = len(set().union(*train_sets))
# simple_matching = ...

*Отговор за задача 1:*

## Задача 2. Матрица на разстоянията между кварталите

В упражнението обявите получиха само разстоянието до центъра. Тук всеки
квартал е точка с две координати в километри, `east_km` и `north_km`.

1. Изчислете матрицата на евклидовите разстояния между 22-та квартала.
   Проверете, че е симетрична и че по диагонала има нули. Кои свойства на
   разстоянието от лекцията се виждат така?
2. За всеки квартал намерете най-близкия друг квартал.
3. Намерете две двойки квартали, които са почти еднакво далеч от центъра,
   но далеч един от друг. Какво губи представянето само с разстоянието до
   центъра?

<details><summary>Подсказка</summary>

`pairwise_distances(points, metric="euclidean")` от `sklearn.metrics` връща
цялата матрица. За да не излезе самият квартал като най-близък, сложете
безкрайност по диагонала с`np.fill_diagonal(distances, np.inf)` и използвайте
`argmin(axis=1)`. Разстоянието до центъра е `np.hypot(east_km, north_km)`.

</details>

In [ ]:
# points = neighborhoods[["east_km", "north_km"]].to_numpy()
# names = neighborhoods["neighborhood"].to_numpy()
# euclidean = pairwise_distances(points, metric="euclidean")
# print("Симетрична:", ..., "| нули по диагонала:", ...)
# ...
# radius = np.hypot(points[:, 0], points[:, 1])

*Отговор за задача 2:*

## Задача 3. Ординален признак: времето навън

Колоната `weather` в `rides` описва времето с четири стойности: "ясно",
"облачно", "слаб валеж" и "силен валеж". Те имат естествен ред, т.е.
признакът е **ординален**, като "малък", "среден" и "голям".
`rides_train` и `rides_test` са разделени по време, както в упражнението.

1. Колко часа има от всеки вид време и колко велосипеда се наемат средно?
   Личи ли редът от данните?
2. Кодирайте `weather` с `OrdinalEncoder()` без параметри и покажете
   `categories_`. В какъв ред са категориите и какво не е наред? Кодирайте
   отново с правилния ред.
3. Обучете линейни модели с часа като 24 one-hot колони, `workingday` и
   `year`, плюс времето: (а) с правилния ординален код, (б) с кода по
   подразбиране и (в) като 4 one-hot колони. Сравнете средната грешка върху
   `rides_test` и коефициентите пред времето.
4. Ординалният код предполага, че всяка следваща степен променя наемането с
   едно и също число. Потвърждават ли това коефициентите на one-hot модела?
   Колко часа "силен валеж" има в обучението и колко може да се вярва на
   коефициента му?

<details><summary>Подсказка</summary>

`rides.groupby("weather")["count"].agg(["size", "mean"])` дава броя и
средното. Правилният ред се задава с
`OrdinalEncoder(categories=[["ясно", "облачно", "слаб валеж", "силен валеж"]])`.
За часа използвайте `OneHotEncoder(categories=[list(range(24))], sparse_output=False)`,
а за времето като one-hot добавете и `handle_unknown="ignore"`. Колоните се
събират с `np.c_[...]`.

</details>

In [ ]:
# print(rides.groupby("weather")["count"].agg(["size", "mean"]))
# default_encoder = OrdinalEncoder().fit(rides_train[["weather"]])
# print(default_encoder.categories_)
# ordinal_encoder = ...
# hour_encoder = OneHotEncoder(categories=[list(range(24))], sparse_output=False).fit(rides_train[["hour"]])
# for name, encoder in ...:
#     ...

*Отговор за задача 3:*

## Задача 4. Липсваща година на строителство

В част 3 на упражнението подробностите за обявите бяха заредени от
`data/sofia_apartment_details.json`, но годината на строителство остана
настрана, защото липсва в част от обявите.

1. Заредете файла и го свържете с `located`, както в упражнението. В колко
   обяви липсва годината? Сравнете средната цена на квадратен метър на
   обявите с година и без година. Каква може да е причината годината да
   липсва?
2. Изчислете възрастта на сградата: 2026 минус годината. Разделете обявите,
   както в упражнението (`test_size=0.2`, `random_state=42`) и обучете два
   модела с площта, разстоянието и възрастта: (а) липсващата възраст е
   попълнена с медианата от обучението; (б) `SimpleImputer(strategy="median",
   add_indicator=True)` добавя и колона с 1 за липсваща възраст. Сравнете
   средната грешка **със знак** (истинската цена минус прогнозата) на
   тестовите обяви без година. Какво означава коефициентът пред индикатора?
3. Сравнете средната абсолютна грешка на двата модела за 20 различни
   разделяния (`random_state` от 0 до 19). В колко от тях индикаторът
   помага? Какво показва това за проверката с едно разделяне?

<details><summary>Подсказка</summary>

`pd.json_normalize` превръща вложеното поле `building` в колони
`building.year_built` и `building.elevator`, а
`located.join(details.set_index("listing"))` свързва двете таблици по
номера на реда. С `add_indicator=True` методът `transform` връща две
колони: попълнената възраст и 0 или 1 за липса.

</details>

In [ ]:
# details = pd.json_normalize(json.load(open(DATA_DIR / "sofia_apartment_details.json", encoding="utf-8")))
# homes = located.join(details.set_index("listing"))
# missing = homes["building.year_built"].isna()
# ...
# homes["age"] = ...
# homes_train, homes_test = train_test_split(homes, test_size=0.2, random_state=42)
# for add_indicator in [False, True]:
#     imputer = SimpleImputer(strategy="median", add_indicator=add_indicator).fit(...)
#     ...

*Отговор за задача 4:*